# Lab 2 — KNN Classifier (Loan Default)

**Day 04 · Distance-Based ML & MLOps · Cisco AI/ML Training**

---

## Learning objectives

1. Build a **scikit-learn Pipeline** with `StandardScaler` before `KNeighborsClassifier`.
2. Train a KNN model with **k = 5** on scaled numeric loan features.
3. Evaluate with **test accuracy** and inspect sample predictions.
4. Compare KNN (~0.55) to Day 3 logistic regression (~0.59).

> **Checkpoints:** train **800** / test **200** · k = **5** · accuracy ≈ **0.55** · preds include **0** and **1**

**Companion script:** `../scripts/lab02_knn_classifier.py`

## K-Nearest Neighbors in one slide

KNN is a **lazy** learner — it stores training data and classifies by **majority vote** among the closest training points.

| Step | What happens |
|------|--------------|
| 1. Scale | Put every feature on comparable units (`StandardScaler`) |
| 2. Distance | Measure how far a test loan is from each training loan |
| 3. Vote | Pick the **k** nearest neighbors; majority class wins |

**Why scaling?** Without it, large-magnitude features (e.g. `annual_inc`) dominate distance — see Lab 1.

**vs Logistic Regression (Day 3):** KNN makes no linear assumption; it can capture local patterns but is sensitive to irrelevant features and k.

---

## 1. Load data and select features

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-04":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "lending-club" / "lending_club_sample.csv").is_file():
            GH_ROOT = parent
            break

DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]

df = pd.read_csv(GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv")
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

X = df[NUMERIC_FEATURES]
y = df["default"]

print(f"rows: {len(df)}")
print(f"default rate: {y.mean():.4f}")
display(X.head(3))

---

## 2. Train/test split with stratification

`stratify=y` keeps the same default rate in train and test — same split settings as Day 3 for fair comparison.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train size: {len(X_train)}")
print(f"test size:  {len(X_test)}")
print(f"default rate (train): {y_train.mean():.4f}")
print(f"default rate (test):  {y_test.mean():.4f}")

assert len(X_train) == 800 and len(X_test) == 200

---

## 3. Pipeline: scale → KNN (k = 5)

The pipeline ensures scaling is **fit on train only** and applied consistently at predict time.

In [ ]:
K = 5

pipe = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=K)),
    ]
)

print(pipe)

---

## 4. Fit and evaluate

In [ ]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Lab 2 — KNN classifier")
print(f"k (neighbors): {K}")
print(f"test accuracy: {accuracy:.4f}")
print(f"sample predictions (first 5): {y_pred[:5].tolist()}")

### Sample predictions vs actual

KNN outputs hard labels (0 or 1) — no probability unless you add `predict_proba` (available on `KNeighborsClassifier`).

In [ ]:
sample = pd.DataFrame({
    "int_rate": X_test.iloc[:5]["int_rate"].values,
    "dti": X_test.iloc[:5]["dti"].values,
    "predicted": y_pred[:5],
    "actual": y_test.iloc[:5].values,
})
display(sample)

unique_preds = set(y_pred.tolist())
print(f"unique predicted labels: {sorted(unique_preds)}")
assert 0 in unique_preds and 1 in unique_preds

---

## 5. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = ["no default (0)", "default (1)"]

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"KNN confusion matrix (k={K}, acc={accuracy:.2f})")
plt.tight_layout()
plt.show()

Accuracy ≈ **0.55** is modest — KNN with k=5 is a useful baseline, but Lab 3 will show whether a different **k** improves it.

---

## 6. Compare to Day 3 logistic regression

| Model | Typical test accuracy (this sample) |
|-------|-------------------------------------|
| Logistic regression (Day 3) | ~**0.59** |
| KNN k=5 (this lab) | ~**0.55** |

Linear models can outperform KNN when the decision boundary is roughly linear and features are well chosen. Lab 3 sweeps **k** to find a better neighbor count.

---

## 7. Checkpoint summary

In [ ]:
assert len(X_train) == 800 and len(X_test) == 200
assert K == 5
assert abs(accuracy - 0.55) < 0.02
assert y_pred[:5].tolist() == [1, 0, 1, 0, 1]
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Why must `StandardScaler` come **before** KNN in the pipeline?
2. What happens if k=1? What if k equals the full training set size?
3. Would adding categorical features (grade, term) help without encoding?

**Previous:** [Lab 1 — Distance metrics](lab01_distance_metrics.ipynb)  
**Next:** [Lab 3 — Choose K](lab03_choose_k.ipynb)